In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.nem_battery.load import load_raw

df = load_raw("SA1")

Found 60 files


### 1. EDA 

In [2]:
# check nulls 
print(df.isnull().sum())

region             0
settlement_date    0
total_demand_mw    0
rrp                0
period_type        0
interval_start     0
dtype: int64


In [3]:
# category values
print(df["region"].unique())
print(df["period_type"].value_counts())

<ArrowStringArray>
['SA1']
Length: 1, dtype: str
period_type
TRADE    525888
Name: count, dtype: int64


In [4]:
# price range and negative prices 
print(df["rrp"].describe())

neg = (df["rrp"] < 0)
print(f"Negative intervals: {neg.sum():,} ({neg.mean():.1%} of all intervals)")
print("Lowest price:", df["rrp"].min(), "at", df.loc[df["rrp"].idxmin(), "settlement_date"])
print("Highest price:", df["rrp"].max(), "at", df.loc[df["rrp"].idxmax(), "settlement_date"])

count    525888.000000
mean         98.737607
std         479.997943
min       -1000.000000
25%           0.000000
50%          67.575000
75%         136.040000
max       20300.000000
Name: rrp, dtype: float64
Negative intervals: 129,682 (24.7% of all intervals)
Lowest price: -1000.0 at 2021-10-13 08:45:00
Highest price: 20300.0 at 2026-01-18 19:30:00


In [5]:
neg = (df["rrp"] < 0)   # True/False for each interval
neg.sum()               # count of True = 129,682
neg.mean()              # share of True = 24.7%


np.float64(0.24659623341852258)

In [6]:
# demand range 
print(df["total_demand_mw"].describe())
print("Zero or negative demand:", (df["total_demand_mw"] <= 0).sum())

count    525888.000000
mean       1295.163786
std         438.277963
min        -311.100000
25%        1106.020000
50%        1354.150000
75%        1544.132500
max        3326.910000
Name: total_demand_mw, dtype: float64
Zero or negative demand: 1222


In [7]:
# rows per month 
per_month = df.groupby(df["interval_start"].dt.to_period("M")).size()
print(per_month.value_counts())

8928    35
8640    20
8064     4
8352     1
Name: count, dtype: int64


Summary: 
- The average price ~ $98.74 and the median $67.58 -> the mean is 46% higher than the median. Rare, huge spikes drag the mean up, so the data is right-skewed.
- Standard deviation $480: about 5x the mean, which means extreme volatility
- Negative prices $24.7 of intervals: about 1 in 4 intervals in SA had a negative price 
- Minium price: -$1000 on 13/10/2021 8:45 market price floor 
- Maximum price: $20300 on 18/10/2026 19:30 market price gap 
- Negative interval: 129,628 intervals = 450 days worth spread over 5 years = 90 days worth per year = 6 hours of every day had negative prices
- The median demand around 1295.16 but maximum touches 3326.9 MW, likely a hot summer evening with air con (assumption) -> check later. The minimum is -311MW. The total demand could be negative: solar produces more than demand.


#### Investigation of negative prices

In [8]:
print(df.loc[df["total_demand_mw"].idxmin(), ["settlement_date", "total_demand_mw", "rrp"]])
print(df.loc[df["total_demand_mw"].idxmax(), ["settlement_date", "total_demand_mw", "rrp"]])

settlement_date    2025-12-25 13:35:00
total_demand_mw                 -311.1
rrp                            -299.67
Name: 445410, dtype: object
settlement_date    2025-02-12 18:55:00
total_demand_mw                3326.91
rrp                           14500.44
Name: 354466, dtype: object


Christmas Day at 13:35, assume business closed, low consumption, high supply due to summer sun. Demand went negative and generators were paying almosnt $300/MWh to keep running.

However, the peak around February is quite abnormal.

In [9]:
# save Parquet
from src.nem_battery.load import INTERIM_DIR

# 1. Make sure the folder exists
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# 2. Save
out = INTERIM_DIR / "price_demand_sa1.parquet"
df.to_parquet(out, index=False)

# 3. Prove it saved correctly: read it back and compare
check = pd.read_parquet(out)
print(check.dtypes)
print("Same rows:", len(check) == len(df))
print(f"Parquet size: {out.stat().st_size / 1e6:.1f} MB")

region                        str
settlement_date    datetime64[us]
total_demand_mw           float64
rrp                       float64
period_type                   str
interval_start     datetime64[us]
dtype: object
Same rows: True
Parquet size: 10.5 MB
